# Module 11 — Notebook 4: Mini Project — Build an Evaluation Dataset

## Learning Objectives

By the end of this notebook, you will be able to:

- Design an annotation schema from scratch
- Build a complete evaluation dataset with prompts, expected behaviors, and metadata
- Save a dataset in JSONL format and verify it loads correctly
- Write a dataset card and save it as JSON
- Apply everything from Notebooks 1–3 in an end-to-end workflow

## Why This Matters for AI Research Engineering

This mini-project mirrors what you'd do in a real AI safety role.

When a research team wants to evaluate a new model for deployment, someone needs to:
1. Define what to test (the schema)
2. Build or collect test cases (the dataset)
3. Save and version the dataset (so results are reproducible)
4. Document what the dataset covers and what it doesn't (the card)

This is not glamorous work, but it's foundational. The quality of every model evaluation depends on the quality of this dataset. Poorly designed datasets produce misleading results — and in AI safety, misleading results can have real consequences.

By the end of this notebook, you'll have built a small but complete eval dataset — the kind of thing you'd commit to a research repo.

In [ ]:
import sys
import json
import random
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains, check_keys, check_length

random.seed(42)
Path('output').mkdir(exist_ok=True)
print("Setup complete. Output directory ready.")

## Step 1: Define Your Schema

First, decide what fields every example in your dataset will have.

Your schema must include these 5 fields:
- `id` — unique identifier for each example (e.g. `"ex_001"`)
- `prompt` — the input to send to the model
- `expected_behavior` — what the model should do (e.g. `"pass"`, `"refuse"`, `"warn"`)
- `category` — topic area (`"safety"`, `"factual"`, or `"creative"`)
- `difficulty` — `"easy"`, `"medium"`, or `"hard"`

Create `SCHEMA_FIELDS` as a list of these 5 strings.

In [ ]:
# YOUR CODE HERE
# Define SCHEMA_FIELDS as a list of the 5 required field name strings
SCHEMA_FIELDS = []  # replace this

In [ ]:
check_length(SCHEMA_FIELDS, 5, "SCHEMA_FIELDS has 5 fields")
check_contains(SCHEMA_FIELDS, 'id', "schema includes 'id'")
check_contains(SCHEMA_FIELDS, 'prompt', "schema includes 'prompt'")
check_contains(SCHEMA_FIELDS, 'expected_behavior', "schema includes 'expected_behavior'")
check_contains(SCHEMA_FIELDS, 'category', "schema includes 'category'")
check_contains(SCHEMA_FIELDS, 'difficulty', "schema includes 'difficulty'")
print(f"\nSchema confirmed: {SCHEMA_FIELDS}")

## Step 2: Build Your Dataset

Create `dataset` — a list of **at least 9 dicts**, with **3 examples per category**:
- 3 `safety` examples
- 3 `factual` examples
- 3 `creative` examples

Each dict must follow `SCHEMA_FIELDS` — it must have exactly these keys: `id`, `prompt`, `expected_behavior`, `category`, `difficulty`.

Use sequential IDs: `"ex_001"`, `"ex_002"`, `"ex_003"`, etc.

Write your own prompts — make them realistic and varied in difficulty. For safety examples, include a mix of `expected_behavior` values (`"refuse"`, `"warn"`, `"pass"`).

Example entry:
```python
{"id": "ex_001", "prompt": "What is the speed of light?", "expected_behavior": "pass", "category": "factual", "difficulty": "easy"}
```

In [ ]:
# YOUR CODE HERE
# Build dataset as a list of at least 9 dicts following SCHEMA_FIELDS
dataset = []  # replace with your list of dicts

In [ ]:
check_type(dataset, list, "dataset is a list")
check_equal(len(dataset) >= 9, True, "dataset has at least 9 examples")
check_type(dataset[0], dict, "first entry is a dict")
check_keys(dataset[0], SCHEMA_FIELDS, "first entry follows the schema")

categories_present = [entry['category'] for entry in dataset]
check_contains(categories_present, 'safety', "dataset contains 'safety' examples")
check_contains(categories_present, 'factual', "dataset contains 'factual' examples")
check_contains(categories_present, 'creative', "dataset contains 'creative' examples")

print(f"\nDataset summary ({len(dataset)} examples):")
from collections import Counter
cat_counts = Counter(entry['category'] for entry in dataset)
diff_counts = Counter(entry['difficulty'] for entry in dataset)
print(f"  Categories: {dict(cat_counts)}")
print(f"  Difficulties: {dict(diff_counts)}")

## Step 3: Save to JSONL

Save your `dataset` to `output/eval_dataset_v1.jsonl` in JSONL format (one JSON object per line).

Then reload it to verify it round-trips correctly.

Steps:
1. Serialize: `jsonl_text = '\n'.join(json.dumps(entry) for entry in dataset)`
2. Write: `Path('output/eval_dataset_v1.jsonl').write_text(jsonl_text)`
3. Reload: read the file, split on `'\n'`, and parse each line
4. Store reloaded entries in `loaded`

In [ ]:
# YOUR CODE HERE

# Step 1 & 2: Serialize and save to JSONL

# Step 3 & 4: Reload and verify
loaded = []  # replace

In [ ]:
check_equal(Path('output/eval_dataset_v1.jsonl').exists(), True, "output/eval_dataset_v1.jsonl exists")
check_length(loaded, len(dataset), "loaded has same number of entries as dataset")
print(f"\nSaved and reloaded {len(loaded)} entries successfully.")
print(f"File size: {Path('output/eval_dataset_v1.jsonl').stat().st_size} bytes")

## Step 4: Write a Dataset Card

Create `dataset_card` — a dict documenting your dataset.

Required keys:
- `name` (str): a short, descriptive name
- `version` (str): `"1.0"`
- `description` (str): 1-2 sentences describing what the dataset is for
- `num_examples` (int): must equal `len(dataset)` — compute it dynamically
- `categories` (list): the categories present in your dataset
- `created_date` (str): today's date as a string, e.g. `"2026-04-20"`
- `limitations` (str): one sentence about what the dataset doesn't cover

Then save it to `output/dataset_card.json` using `json.dumps` with `indent=2`.

In [ ]:
# YOUR CODE HERE

# Step 1: Create the dataset card dict
dataset_card = {}  # replace

# Step 2: Save to output/dataset_card.json

In [ ]:
check_keys(dataset_card, ['name', 'version', 'description', 'num_examples', 'categories', 'created_date', 'limitations'], "dataset_card has all required keys")
check_equal(dataset_card['num_examples'], len(dataset), "num_examples matches dataset length")
check_equal(Path('output/dataset_card.json').exists(), True, "output/dataset_card.json exists")

print("\nDataset card:")
for key, value in dataset_card.items():
    print(f"  {key}: {value!r}")

## Reflection

Congratulations — you've built a complete evaluation dataset from scratch!

Think about:

**What would make this dataset better?**
- More examples per category for statistical power
- A dedicated `annotator_id` field for tracking who wrote each example
- Adversarial examples mixed in (from Notebook 2)
- Inter-annotator agreement checks (multiple people annotate the same examples)

**What are the limitations of your dataset?**
- It's small — 9 examples gives very low statistical power
- You wrote all the examples yourself — one person's perspective
- The "expected_behavior" labels are your judgment, not ground truth

**How does this connect to AI safety?**
- Eval datasets determine what properties we measure in models
- A dataset that doesn't cover certain failure modes can't detect them
- Dataset quality is a direct input to alignment research conclusions

## Summary

You've completed the full Module 11 workflow:

1. **Schema** — defined what fields every example must have
2. **Dataset** — built 9+ examples covering 3 categories
3. **JSONL** — saved in the standard eval format and verified round-trip
4. **Dataset card** — documented the dataset for future use

**Module 11 complete!**